In [25]:
import nltk
import random
import string
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk import FreqDist

# Pobieranie zasobów
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mkowa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mkowa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mkowa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\mkowa\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [ ]:


lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Przykładowe recenzje z etykietami
training_data = [
    ("I absolutely loved this movie. It was heartwarming and beautifully filmed.", "pos"),
    ("The plot was weak and the characters were incredibly dull.", "neg"),
    ("Such an inspiring story. I cried and laughed all the way through.", "pos"),
    ("I regret wasting two hours of my life on this garbage.", "neg"),
    ("Brilliant screenplay and excellent performances by the entire cast.", "pos"),
    ("It was confusing, boring, and visually unappealing.", "neg"),
    ("An unforgettable experience. One of the best movies of the year!", "pos"),
    ("The dialogue was cheesy and the pacing was horrible.", "neg"),
    ("What a masterpiece! It touched my soul deeply.", "pos"),
    ("Terrible acting and a complete lack of originality.", "neg"),
    ("A delightful film full of charm and great humor.", "pos"),
    ("This was just plain bad. I couldn’t stand it.", "neg"),
    ("Incredible storytelling and a powerful emotional journey.", "pos"),
    ("Nothing made sense. The story jumped all over the place.", "neg"),
    ("The visuals were breathtaking and the music was perfect.", "pos"),
    ("The worst film I’ve seen this decade.", "neg"),
    ("A refreshing take on an old theme. Truly enjoyable.", "pos"),
    ("The ending made no sense and left me angry.", "neg"),
    ("It’s one of those rare films that stays with you long after it's over.", "pos"),
    ("I walked out halfway through — that says it all.", "neg")
]

def clean_words(text):
    tokens = nltk.word_tokenize(text.lower())
    tokens = [t.translate(str.maketrans('', '', string.punctuation)) for t in tokens]
    return [lemmatizer.lemmatize(t) for t in tokens if t and t not in stop_words]


documents = [(clean_words(text), label) for (text, label) in training_data]

all_words = FreqDist([word for doc, _ in documents for word in doc])
word_features = list(all_words)[:100]  


def document_features(document):
    doc_words = set(document)
    return {word: (word in doc_words) for word in word_features}


featuresets = [(document_features(d), c) for (d, c) in documents]
random.shuffle(featuresets)

train_set = featuresets 
classifier = nltk.NaiveBayesClassifier.train(train_set)

# 🔍 Testowe zdania
test_reviews = [
    "Absolutely loved it. One of the best films!",
    "Boring and slow. I didn’t enjoy it at all.",
    "It was okay. Nothing special though.",
    "Amazing visuals and touching story.",
    "it was so beautifull, i want watch it again! i love this kind of movie",
    "best movie i ever seen",
    "it was really good movie",
    "really? what was that?"
]

print("🧪 Predykcje na testowych recenzjach:\n")
for rev in test_reviews:
    cleaned = clean_words(rev)
    features = document_features(cleaned)
    prediction = classifier.classify(features)
    print(f"→ {rev}\n   ➤ Predykcja: {prediction}\n")

🧪 Predykcje na testowych recenzjach:

→ Absolutely loved it. One of the best films!
   ➤ Predykcja: pos

→ Boring and slow. I didn’t enjoy it at all.
   ➤ Predykcja: neg

→ It was okay. Nothing special though.
   ➤ Predykcja: neg

→ Amazing visuals and touching story.
   ➤ Predykcja: pos

→ it was so beautifull, i want watch it again! i love this kind of movie
   ➤ Predykcja: pos

→ best movie i ever seen
   ➤ Predykcja: pos

→ it was really good movie
   ➤ Predykcja: pos

→ really? what was that?
   ➤ Predykcja: neg



In [ ]:


sia = SentimentIntensityAnalyzer()

test_reviews = [
    # Pozytywne
    "Absolutely loved it. One of the best films!",
    "Amazing visuals and touching story.",
    "It was so beautiful, I want to watch it again! I love this kind of movie.",
    "Best movie I’ve ever seen.",
    "It was a really good movie.",
    "This movie made me laugh and cry. Fantastic.",
    "Wonderful experience, truly heartwarming.",
    "Masterpiece. Nothing less.",
    "So much fun to watch!",
    "Exceeded all my expectations. Brilliant!",

    # Neutralne
    "It was okay. Nothing special though.",
    "Not bad, not great either.",
    "I guess it was fine for a lazy evening.",
    "Just another average movie.",
    "The film was neither exciting nor boring.",
    "It had some moments, but overall... meh.",
    "Mediocre but watchable.",
    "Story was predictable, but acting was decent.",
    
    # Negatywne
    "Boring and slow. I didn’t enjoy it at all.",
    "Really? What was that?",
    "Waste of time. Totally overrated.",
    "Poor script and terrible acting.",
    "I walked out halfway through. That bad.",
    "Worst movie I’ve seen this year.",
    "Painfully dull from start to finish.",
    "So much hype for nothing.",
    "The plot made no sense at all.",
    "Fell asleep twice. That says enough.",
    
    # Sugerujący ironię lub niejednoznaczne
    "Wow. Just wow... (not in a good way)",
    "Well, that was... something.",
    "10/10 if you enjoy torturing yourself.",
    "Sure, if you like movies with no plot and random explosions.",
    "Beautifully bad.",
]


for revs in test_reviews:
    scores = sia.polarity_scores(revs)
    
    print(revs, end=": ")

    compound = scores['compound']
    if compound >= 0.05:
        print("POZYTYWNY", end=" ")
    elif compound <= -0.05:
        print("NEGATYWNY", end=" ")
    else:
        print("NEUTRALNY", end=" ")

    print(scores)


Absolutely loved it. One of the best films!: POZYTYWNY {'neg': 0.0, 'neu': 0.409, 'pos': 0.591, 'compound': 0.8653}
Amazing visuals and touching story.: POZYTYWNY {'neg': 0.0, 'neu': 0.513, 'pos': 0.487, 'compound': 0.5859}
It was so beautiful, I want to watch it again! I love this kind of movie.: POZYTYWNY {'neg': 0.0, 'neu': 0.499, 'pos': 0.501, 'compound': 0.9014}
Best movie I’ve ever seen.: POZYTYWNY {'neg': 0.0, 'neu': 0.488, 'pos': 0.512, 'compound': 0.6369}
It was a really good movie.: POZYTYWNY {'neg': 0.0, 'neu': 0.556, 'pos': 0.444, 'compound': 0.4927}
This movie made me laugh and cry. Fantastic.: POZYTYWNY {'neg': 0.203, 'neu': 0.327, 'pos': 0.471, 'compound': 0.6249}
Wonderful experience, truly heartwarming.: POZYTYWNY {'neg': 0.0, 'neu': 0.093, 'pos': 0.907, 'compound': 0.8658}
Masterpiece. Nothing less.: POZYTYWNY {'neg': 0.0, 'neu': 0.328, 'pos': 0.672, 'compound': 0.6249}
So much fun to watch!: POZYTYWNY {'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.5954}
Excee

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\mkowa\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
